In [1]:
import pandas as pd
import numpy as np

In [44]:
pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_rows', None)


In [ ]:
import re
from io import StringIO

# Read and clean the file content
with open(file_path, 'r') as f:
    content = f.read()

# Remove spaces between closing quote and delimiter: "Value"   | -> "Value"|
cleaned_content = re.sub(r'"\s+\|', '"|', content)

# Parse the cleaned content
df = pd.read_csv(StringIO(cleaned_content), sep='|', skipinitialspace=True, engine='python')
df.columns = df.columns.str.strip()

print("Data loaded successfully.")


In [9]:
# Display first few rows
df.head()

,create_at,type,counterparty,Product country,Product ID,Product name,Item ID,Item name,Amount,Currency,Qty
0,2026-02-01 03:59:59.674247+04,DTU_PURCHASE,USD | Xsolla,CIS,441,Steam Wallet Top Up | CIS,3143,Steam Wallet Top Up | CIS,3.2443,USD,1
1,2026-02-01 03:59:59.674247+04,DTU_SALE,USDT | Dessley,CIS,441,Steam Wallet Top Up | CIS,3143,Steam Wallet Top Up | CIS,3.2955,USD,1
2,2026-02-01 03:59:58.815653+04,DTU_PURCHASE,USD | Xsolla,CIS,441,Steam Wallet Top Up | CIS,3143,Steam Wallet Top Up | CIS,0.0950,USD,1
3,2026-02-01 03:59:58.815653+04,DTU_SALE,USDT | Dessley,CIS,441,Steam Wallet Top Up | CIS,3143,Steam Wallet Top Up | CIS,0.0965,USD,1
4,2026-02-01 03:59:57.907522+04,DTU_SALE,USDT | Dessley,CIS,441,Steam Wallet Top Up | CIS,3143,Steam Wallet Top Up | CIS,0.0965,USD,1


In [16]:
# Basic info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3490199 entries, 0 to 3490198
Data columns (total 11 columns):
 #   Column           Dtype  
---  ------           -----  
 0   create_at        str    
 1   type             str    
 2   counterparty     str    
 3   Product country  str    
 4   Product ID       int64  
 5   Product name     str    
 6   Item ID          int64  
 7   Item name        str    
 8   Amount           float64
 9   Currency         str    
 10  Qty              int64  
dtypes: float64(1), int64(3), str(7)
memory usage: 292.9 MB


In [28]:
df['create_at'] = pd.to_datetime(df['create_at'])
df['created_at_ymd'] = df['create_at'].dt.date
df['created_at_ym'] = df['create_at'].dt.to_period('M')

/var/folders/1b/7_zgcknn10j782nt1hr38p8w0000gr/T/ipykernel_14344/186920313.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['created_at_ym'] = df['create_at'].dt.to_period('M')


In [ ]:
dfusd = df[df["Currency"].isin(["USD", "USDT", "USDC"])]# 1. Сначала подготавливаем агрегированную таблицу
df

,create_at,type,counterparty,Product country,Product ID,Product name,Item ID,Item name,Amount,Currency,Qty,created_at_ymd,created_at_ym
0,2026-02-01 03:59:59.674247+04:00,DTU_PURCHASE,USD | Xsolla,CIS,441,Steam Wallet Top Up | CIS,3143,Steam Wallet Top Up | CIS,3.24,USD,1,2026-02-01,2026-02
1,2026-02-01 03:59:59.674247+04:00,DTU_SALE,USDT | Dessley,CIS,441,Steam Wallet Top Up | CIS,3143,Steam Wallet Top Up | CIS,3.30,USD,1,2026-02-01,2026-02
2,2026-02-01 03:59:58.815653+04:00,DTU_PURCHASE,USD | Xsolla,CIS,441,Steam Wallet Top Up | CIS,3143,Steam Wallet Top Up | CIS,0.10,USD,1,2026-02-01,2026-02
3,2026-02-01 03:59:58.815653+04:00,DTU_SALE,USDT | Dessley,CIS,441,Steam Wallet Top Up | CIS,3143,Steam Wallet Top Up | CIS,0.10,USD,1,2026-02-01,2026-02
4,2026-02-01 03:59:57.907522+04:00,DTU_SALE,USDT | Dessley,CIS,441,Steam Wallet Top Up | CIS,3143,Steam Wallet Top Up | CIS,0.10,USD,1,2026-02-01,2026-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3490194,2026-01-01 04:00:02.714076+04:00,VOUCHER_PURCHASE,USD | Content Card,US,62,Steam Wallet Code | US,395,$25 Steam Wallet Code,23.75,USD,1,2026-01-01,2026-01
3490195,2026-01-01 04:00:01.996405+04:00,DTU_PURCHASE,USD | Xsolla,CIS,441,Steam Wallet Top Up | CIS,3143,Steam Wallet Top Up | CIS,0.10,USD,1,2026-01-01,2026-01
3490196,2026-01-01 04:00:01.996405+04:00,DTU_SALE,USDT | Dessley,CIS,441,Steam Wallet Top Up | CIS,3143,Steam Wallet Top Up | CIS,0.10,USD,1,2026-01-01,2026-01
3490197,2026-01-01 04:00:01.816653+04:00,VOUCHER_PURCHASE,AED | LikeCard,OM,497,Ooredoo Voucher | OM,3449,OMR 2 Ooredoo Prepaid Voucher,18.74,AED,1,2026-01-01,2026-01


In [30]:
df["created_at_ym"].value_counts()

created_at_ym
2026-01    3483533
2026-02       6666
Freq: M, Name: count, dtype: int64

In [ ]:
df[df["created_at_ym"] == "2026-02"]

In [48]:
df[df["created_at_ym"] == "2026-01"].groupby(["Product name","Item name"]).agg(
    Sum_Amount=("Amount", "sum"), 
    Mean_Amount=("Amount", "mean"), 
    Currency = ("Currency", "first"),
    Quantity=("Qty", "sum")  # Ensure this column name matches your data (e.g., 'Quantity' or 'Qty')
).sort_values(by="Quantity", ascending=False).reset_index().head(50)

,Product name,Item name,Sum_Amount,Mean_Amount,Currency,Quantity
0,Steam Wallet Top Up | CIS,Steam Wallet Top Up | CIS,"18,619,549.47",8.31,USD,2240626
1,PlayStation®Store Wallet | TR,TRY 250 PlayStation®Store Wallet gift card,"1,173,091.99",21.76,USD,65846
2,PlayStation®Store Wallet | TR,TRY 500 PlayStation®Store Wallet gift card,"795,027.97",19.92,USD,49296
3,PlayStation®Store Wallet | US,$10 PlayStation®Store Wallet gift card,"523,616.84",21.72,USD,31713
4,PUBG Mobile Top Up,PUBG Mobile: 60 UC,"27,020.15",0.88,USDT,30798
5,du eVoucher | AE,AED 30 du eVoucher,"899,195.36",54.02,AED,30639
6,PlayStation®Store Wallet | TR,TRY 1000 PlayStation®Store Wallet gift card,"1,119,422.16",53.85,USD,26676
7,App Store & iTunes Code | RU,RUB 500 App Store & iTunes Gift Code,"173,507.55",87.54,EUR,25981
8,Apple Gift Card | US,$2 Apple Gift Card,"42,878.64",2.21,USD,22610
9,PUBG Mobile Gift Card,PUBG Mobile: 60 UC,"23,668.29",2.78,AED,22306


In [ ]:

grouped_df = dfusd[dfusd["created_at_ym"] == "2026-01"].groupby(["Product name", "Item name"]).agg(
    Sum_Amount=("Amount", "sum"), 
    Mean_Amount=("Amount", "mean"), 
    Currency=("Currency", "first"),
    Quantity=("Qty", "sum")
).reset_index()

# 2. Задаем диапазоны (границы) и подписи
# 0-10, 10-100, 100-1000, 1000-20000, 20000+
bins = [0, 10, 100, 1000, 20000, float('inf')]
labels = ['до 10', '10-100', '100-1000', '1000-20000', 'выше 20т']

# 3. Разбиваем колонку Quantity на категории
# right=False означает, что левая граница включена, а правая нет (например, 10 попадет в '10-100')
grouped_df['Range'] = pd.cut(grouped_df['Quantity'], bins=bins, labels=labels, right=False)

summary_table = grouped_df.groupby('Range').agg(
    Count=('Product name', 'count'),       # Количество товаров в диапазоне
    Total_Sum=('Sum_Amount', 'sum')        # Общая сумма продаж в диапазоне
)

summary_table

            Count     Total_Sum
Range                          
до 10        1094    215,532.09
10-100       1099  1,785,544.38
100-1000      562 11,292,182.24
1000-20000    270 43,772,173.01
выше 20т       11 23,512,456.80
